# CRIMAC Pipeline — Raw Files to Training-Ready Dataset

This notebook converts Simrad EK80 `.raw` files into a standardized zarr format
with calibrated S_v and precomputed bottom detection masks, using the CRIMAC
preprocessing and bottom detection pipeline throughout.

The workflow:
1. Run the CRIMAC Docker/Singularity preprocessor on a folder of `.raw` files
2. Load the resulting CRIMAC zarr (calibrated S_v, split-beam angles, heave)
3. Run all five CRIMAC bottom detectors and compare them
4. Select the best detector per file and create binary bottom masks
5. Split the merged zarr into per-file standardized `.zarr` stores
6. Quality verification dashboard

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

from raw_sensor_integration.rsi_bottomdetection import (
    simple_bottom_detector,
    edge_bottom_detector,
    angle_bottom_detector,
    combined_bottom_detector,
    dp_offline_bottom_detector,
)
from raw_sensor_integration.rsi_bottomdetection.parameters import Parameters
from raw_sensor_integration.bottom_detection import (
    bottom_to_sample_index, create_bottom_mask,
)
from raw_sensor_integration.data_loader import save_dataset_zarr, load_dataset_zarr

plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['image.cmap'] = 'viridis'

In [ ]:
# ── Paths ──
# Path to the CRIMAC zarr produced by the Docker preprocessor (section 2).
# This is a single zarr that merges all .raw files from the input folder.
CRIMAC_ZARR_PATH = Path.home() / 'Downloads' / 'crimac_output/test.zarr'

# Output directory for per-file standardized zarr stores.
OUTPUT_DIR = REPO_ROOT / 'Datasets/crimac_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Detection parameters ──
params = Parameters(
    minimum_range=10.0,
    offset=0.5,
    threshold_log_sv=-31.0,
    dp_candidate_threshold_log_sv=-36.0,
    dp_max_candidates_per_ping=6,
    dp_stack_radius=3,
    dp_gradient_weight=1.5,
    dp_energy_weight=1.0,
    dp_contrast_weight=1.0,
    dp_slope_penalty=0.15,
    dp_curvature_penalty=0.08,
    dp_gap_penalty=2.5,
    dp_min_confidence=0.35,
    dp_mask_pad_samples=0,
)

# ── Default detector for batch processing ──
# Override per-file in the DETECTOR_CHOICE dict below.
DEFAULT_DETECTOR = 'combined'

## 2. CRIMAC preprocessor

The CRIMAC preprocessor converts raw EK60/EK80 `.raw` files into a calibrated
zarr store with S_v (volume backscattering strength), split-beam angles, heave,
GPS, and transducer draft. It runs as a Docker container (or Singularity on HPC).

Run these commands in a terminal before proceeding. The preprocessor reads all
`.raw` files from the mounted input directory and produces a single merged zarr.

### Docker

```bash
# Pull the image (once)
docker pull crimac/preprocessor

# Step 1 — build ping/distance index (parquet)
docker run --rm \
  -v /path/to/raw_files:/datain:ro \
  -v /path/to/output:/dataout \
  -v /tmp:/workin \
  -e OUTPUT_TYPE=parquet \
  -e MAIN_FREQ=38000 \
  -e MAX_RANGE_SRC=500 \
  -e OUTPUT_NAME=survey_name \
  crimac/preprocessor

# Step 2 — compute calibrated Sv zarr
docker run --rm \
  -v /path/to/raw_files:/datain:ro \
  -v /path/to/output:/dataout \
  -v /tmp:/workin \
  -e OUTPUT_TYPE=zarr \
  -e MAIN_FREQ=38000 \
  -e MAX_RANGE_SRC=500 \
  -e OUTPUT_NAME=survey_name \
  crimac/preprocessor
```

### Singularity (HPC)

```bash
singularity pull crimac_preprocessor.sif docker://crimac/preprocessor

singularity run \
  --bind /path/to/raw_files:/datain,/path/to/output:/dataout,/tmp:/workin \
  --env OUTPUT_TYPE=zarr,MAIN_FREQ=38000,MAX_RANGE_SRC=500,OUTPUT_NAME=survey_name \
  crimac_preprocessor.sif
```

The output is `<OUTPUT_NAME>_sv.zarr` containing calibrated S_v in linear scale,
split-beam angles, heave, pitch, roll, heading, speed, and transducer draft.

In [ ]:
# Verify the CRIMAC zarr exists
assert CRIMAC_ZARR_PATH.exists(), f'CRIMAC zarr not found: {CRIMAC_ZARR_PATH}'
print(f'CRIMAC zarr: {CRIMAC_ZARR_PATH}')

## 3. Load and inspect CRIMAC zarr

In [ ]:
ds = xr.open_zarr(CRIMAC_ZARR_PATH, consolidated=False)
print(ds)
print(f'\nFrequencies: {ds.frequency.values} Hz')
print(f'Pings: {ds.sizes["ping_time"]:,}')
print(f'Range samples: {ds.sizes["range"]:,}')
print(f'Range: 0 – {float(ds.range[-1]):.1f} m')

if 'raw_file' in ds.coords:
    raw_files = np.unique(ds.raw_file.values)
    print(f'\nSource .raw files ({len(raw_files)}):')
    for f in raw_files:
        n_pings = int((ds.raw_file == f).sum())
        print(f'  {f}  ({n_pings} pings)')
else:
    raw_files = np.array(['unknown'])
    print('\nNo raw_file coordinate — single-file zarr')

has_angles = 'angle_alongship' in ds.data_vars
print(f'\nSplit-beam angles available: {has_angles}')

## 4. Visualize echograms per frequency

Convert the linear S_v values to dB and plot each frequency channel.

In [ ]:
n_freq = ds.sizes['frequency']
fig, axes = plt.subplots(n_freq, 1, figsize=(16, 4 * n_freq), sharex=True, squeeze=False)

range_axis = ds.range.values.astype(np.float64)
ping_idx = np.arange(ds.sizes['ping_time'])

for fi in range(n_freq):
    ax = axes[fi, 0]
    sv_linear = ds.sv.isel(frequency=fi).transpose('ping_time', 'range').values
    sv_db = np.where(sv_linear > 0, 10.0 * np.log10(sv_linear), np.nan).T
    freq_hz = float(ds.frequency.values[fi])

    ax.imshow(sv_db, aspect='auto', origin='upper', vmin=-80, vmax=-34,
              extent=[ping_idx[0], ping_idx[-1], range_axis[-1], range_axis[0]])
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{freq_hz/1000:.1f} kHz — shape {sv_db.shape}')

axes[-1, 0].set_xlabel('Ping index')
plt.tight_layout()
plt.show()

## 5. Bottom detection — all five CRIMAC algorithms

Run each detector on the full zarr and compare results. The CRIMAC detectors
operate on the xarray Dataset directly (linear S_v) and return absolute depth
values that include heave and transducer draft.

In [ ]:
b_simple = simple_bottom_detector.detect_bottom(ds, params)
b_edge = edge_bottom_detector.detect_bottom(ds, params)
b_combined = combined_bottom_detector.detect_bottom(ds, params)

# Angle detector needs split-beam angles in the zarr
try:
    b_angle = angle_bottom_detector.detect_bottom(ds, params)
    has_angle_result = True
except Exception as e:
    print(f'Angle detector failed: {e}')
    b_angle = xr.DataArray(
        np.full(ds.sizes['ping_time'], np.nan),
        dims=['ping_time'], coords={'ping_time': ds.ping_time},
        name='bottom_depth')
    has_angle_result = False

# dp_offline returns a Dataset (not DataArray) with extra variables
b_dp = dp_offline_bottom_detector.detect_bottom(ds, params)

print('All detectors complete.')

In [ ]:
# Convert CRIMAC absolute depths to range-from-transducer for plotting.
# CRIMAC returns: depth = heave + transducer_draft + range - offset
# So: range_from_transducer = depth - heave - transducer_draft + offset

draft = ds.transducer_draft
if 'frequency' in draft.dims:
    draft = draft.isel(frequency=0)
draft = draft.squeeze()
heave_draft = (ds.heave + draft).values.astype(np.float64)

range_simple = b_simple.values - heave_draft + params.offset
range_edge = b_edge.values - heave_draft + params.offset
range_angle = b_angle.values - heave_draft + params.offset
range_combined = b_combined.values - heave_draft + params.offset
range_dp = range_axis[b_dp.bottom_index.values.clip(0, len(range_axis) - 1)]

# First-frequency Sv in dB for plotting
sv_linear_0 = ds.sv.isel(frequency=0).transpose('ping_time', 'range').values
sv_db_0 = np.where(sv_linear_0 > 0, 10.0 * np.log10(sv_linear_0), np.nan).T

In [ ]:
# Overlay all detectors on the echogram
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

extent = [ping_idx[0], ping_idx[-1], range_axis[-1], range_axis[0]]

axes[0].imshow(sv_db_0, aspect='auto', origin='upper', vmin=-80, vmax=-34, extent=extent)
axes[0].plot(ping_idx, range_simple, color='cyan', lw=1.0, alpha=0.7, label='Simple')
axes[0].plot(ping_idx, range_edge, color='orange', lw=1.0, alpha=0.7, label='Edge')
if has_angle_result:
    axes[0].plot(ping_idx, range_angle, color='white', lw=1.0, alpha=0.7, label='Angle')
axes[0].plot(ping_idx, range_combined, color='magenta', lw=1.2, alpha=0.9, label='Combined')
axes[0].set_ylabel('Range from transducer (m)')
axes[0].set_title('CRIMAC detectors')
axes[0].legend(loc='upper right', ncol=4)

axes[1].imshow(sv_db_0, aspect='auto', origin='upper', vmin=-80, vmax=-34, extent=extent)
axes[1].plot(ping_idx, range_dp, color='lime', lw=1.4, alpha=0.9, label='DP offline')
axes[1].plot(ping_idx, range_combined, color='magenta', lw=0.8, alpha=0.5, label='Combined (ref)')
axes[1].set_ylabel('Range from transducer (m)')
axes[1].set_xlabel('Ping index')
axes[1].set_title('DP offline (Viterbi)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Per-detector statistics
detectors_depth = {
    'simple': b_simple.values,
    'edge': b_edge.values,
    'angle': b_angle.values,
    'combined': b_combined.values,
    'dp_offline': b_dp.bottom_depth.values,
}
detectors_range = {
    'simple': range_simple,
    'edge': range_edge,
    'angle': range_angle,
    'combined': range_combined,
    'dp_offline': range_dp,
}

rows = []
for name, arr in detectors_range.items():
    valid = np.isfinite(arr)
    rows.append({
        'detector': name,
        'valid_pings': int(valid.sum()),
        'valid_pct': f'{100 * valid.mean():.1f}%',
        'depth_min': f'{np.nanmin(arr):.1f}' if valid.any() else 'N/A',
        'depth_median': f'{np.nanmedian(arr):.1f}' if valid.any() else 'N/A',
        'depth_max': f'{np.nanmax(arr):.1f}' if valid.any() else 'N/A',
    })

pd.DataFrame(rows)

## 6. Select detector and configure per-file choices

Edit the `DETECTOR_CHOICE` dict below after inspecting the results in section 5.
Files not listed use the `DEFAULT_DETECTOR` from section 1.

In [ ]:
# Per-file detector override. Keys are .raw filenames from the raw_file coordinate.
# Values: 'simple', 'edge', 'angle', 'combined', 'dp_offline'
DETECTOR_CHOICE = {
    # 'D20210507-T154515.raw': 'dp_offline',
}

def get_detector_choice(raw_filename):
    return DETECTOR_CHOICE.get(raw_filename, DEFAULT_DETECTOR)

def get_bottom_range_for_detector(detector_name, ping_mask):
    """Get range-from-transducer for the chosen detector on a subset of pings."""
    hd_sub = heave_draft[ping_mask]
    if detector_name == 'dp_offline':
        idx = b_dp.bottom_index.values[ping_mask].clip(0, len(range_axis) - 1)
        return range_axis[idx]
    lookup = {
        'simple': b_simple, 'edge': b_edge,
        'angle': b_angle, 'combined': b_combined,
    }
    depth_vals = lookup[detector_name].values[ping_mask]
    return depth_vals - hd_sub + params.offset

## 7. Batch processing — split into per-file standardized zarrs

The CRIMAC zarr merges all source `.raw` files. This section splits it back
into per-file zarr stores in our standardized format, using the chosen detector
for each file.

In [ ]:
summaries = []
n_freq = ds.sizes['frequency']
frequencies = ds.frequency.values.astype(np.float64)

# Get transducer draft per frequency
td_vals = ds.transducer_draft
if 'ping_time' in td_vals.dims:
    td_scalar = float(td_vals.isel(frequency=0, ping_time=0))
else:
    td_scalar = float(td_vals.isel(frequency=0))

for raw_file in tqdm(raw_files, desc='Saving per-file zarrs'):
    output_path = OUTPUT_DIR / f'{Path(raw_file).stem}.zarr'
    detector_name = get_detector_choice(raw_file)

    # Select pings belonging to this file
    if 'raw_file' in ds.coords:
        ping_mask = (ds.raw_file.values == raw_file)
    else:
        ping_mask = np.ones(ds.sizes['ping_time'], dtype=bool)

    ping_indices = np.where(ping_mask)[0]
    n_pings = len(ping_indices)

    # Extract Sv for all frequencies — shape (n_freq, n_range, n_pings)
    sv_3d = np.empty((n_freq, ds.sizes['range'], n_pings), dtype=np.float32)
    for fi in range(n_freq):
        sv_lin = ds.sv.isel(frequency=fi, ping_time=ping_indices).values  # (n_pings, n_range)
        sv_3d[fi] = np.where(sv_lin > 0, 10.0 * np.log10(sv_lin), np.nan).T

    # Trim trailing NaN range samples (rows that are all NaN across all freqs and pings)
    all_nan_rows = np.all(np.isnan(sv_3d), axis=(0, 2))
    if all_nan_rows.any():
        last_valid = np.where(~all_nan_rows)[0][-1]
        trim_to = min(last_valid + 50, ds.sizes['range'])
        sv_3d = sv_3d[:, :trim_to, :]
        range_trimmed = range_axis[:trim_to]
    else:
        range_trimmed = range_axis

    # Bottom detection (range from transducer)
    bottom_range = get_bottom_range_for_detector(detector_name, ping_mask)

    # Create binary mask using trimmed range
    bottom_indices = bottom_to_sample_index(bottom_range, range_trimmed)
    mask = create_bottom_mask(len(range_trimmed), n_pings, bottom_indices, mode='below')

    # Extract heave
    heave_file = ds.heave.values[ping_indices].astype(np.float32)

    # Extract angles if available
    angle_along = None
    angle_athwart = None
    if has_angles:
        angle_along = np.empty_like(sv_3d)
        angle_athwart = np.empty_like(sv_3d)
        for fi in range(n_freq):
            angle_along[fi] = ds.angle_alongship.isel(
                frequency=fi, ping_time=ping_indices).values.T[:len(range_trimmed)]
            angle_athwart[fi] = ds.angle_athwartship.isel(
                frequency=fi, ping_time=ping_indices).values.T[:len(range_trimmed)]

    # Timestamps
    timestamps = ds.ping_time.values[ping_indices]

    # Channel IDs
    channel_ids = [str(c) for c in ds.channel_id.values] if 'channel_id' in ds.coords else []

    # Save
    save_dataset_zarr(
        sv_db=sv_3d,
        frequencies=frequencies,
        range_array=range_trimmed,
        timestamps=timestamps,
        bottom_range=bottom_range,
        bottom_mask=mask,
        heave=heave_file,
        output_path=output_path,
        angle_along=angle_along,
        angle_athwart=angle_athwart,
        pipeline='crimac',
        detection_algorithm=detector_name,
        detection_params={
            'minimum_range': params.minimum_range,
            'offset': params.offset,
            'threshold_log_sv': params.threshold_log_sv,
        },
        source_files=[str(raw_file)],
        channel_ids=channel_ids,
        transducer_draft=td_scalar,
    )

    valid_pct = 100 * np.mean(np.isfinite(bottom_range))
    summaries.append({
        'file': raw_file,
        'detector': detector_name,
        'n_pings': n_pings,
        'n_freq': n_freq,
        'n_range': len(range_trimmed),
        'valid_pct': f'{valid_pct:.1f}%',
        'depth_median': f'{np.nanmedian(bottom_range):.1f} m',
        'output': output_path.name,
    })

pd.DataFrame(summaries)

## 8. Verification dashboard

Load back the saved zarrs and show echograms with detected bottom lines to
verify everything looks correct at a glance.

In [ ]:
zarr_files = sorted(OUTPUT_DIR.glob('*.zarr'))
n_files = len(zarr_files)
n_cols = min(2, n_files)
n_rows = (n_files + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), squeeze=False)

for idx, zp in enumerate(zarr_files):
    ax = axes[idx // n_cols, idx % n_cols]
    data = load_dataset_zarr(zp)

    # Plot first frequency
    sv = data['sv_db'][0]  # (n_range, n_pings)
    rng = data['range_array']
    br = data['bottom_range']

    ax.imshow(sv, aspect='auto', origin='upper', vmin=-80, vmax=-34,
              extent=[0, sv.shape[1], rng[-1], rng[0]])

    valid = np.isfinite(br)
    ax.plot(np.where(valid)[0], br[valid], 'r-', lw=0.8, alpha=0.9)

    valid_pct = 100 * valid.mean()
    freq_khz = data['frequencies'][0] / 1000
    ax.set_title(f'{zp.name}  ({freq_khz:.0f} kHz, {valid_pct:.0f}% valid)', fontsize=9)
    ax.set_ylabel('Range (m)')

# Hide empty subplots
for idx in range(n_files, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Round-trip verification: load one zarr and check it matches expectations
test_data = load_dataset_zarr(zarr_files[0])
print(f'Loaded: {zarr_files[0].name}')
print(f'  sv_db shape:       {test_data["sv_db"].shape}  (freq, range, pings)')
print(f'  frequencies:       {test_data["frequencies"]} Hz')
print(f'  range:             0 – {test_data["range_array"][-1]:.1f} m')
print(f'  bottom_range:      {np.nanmin(test_data["bottom_range"]):.1f} – {np.nanmax(test_data["bottom_range"]):.1f} m')
print(f'  bottom_mask sum:   {test_data["bottom_mask"].sum():,} pixels')
print(f'  has real angles:   {test_data["attrs"].get("has_angles", False)}')
print(f'  pipeline:          {test_data["attrs"]["pipeline"]}')
print(f'  detector:          {test_data["attrs"]["detection_algorithm"]}')

## 9. Comparison with echopype pipeline

If echopype-processed zarrs exist for the same files (from notebook 14), load
both versions and compare S_v values and bottom detection lines side by side.

In [ ]:
ECHOPYPE_DIR = REPO_ROOT / 'Datasets/echopype_dataset'

if ECHOPYPE_DIR.exists():
    for zp in zarr_files:
        ep_path = ECHOPYPE_DIR / zp.name
        if not ep_path.exists():
            continue

        crimac_data = load_dataset_zarr(zp)
        ep_data = load_dataset_zarr(ep_path)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # CRIMAC echogram + bottom
        sv_c = crimac_data['sv_db'][0]
        rng_c = crimac_data['range_array']
        br_c = crimac_data['bottom_range']
        axes[0].imshow(sv_c, aspect='auto', origin='upper', vmin=-80, vmax=-34,
                       extent=[0, sv_c.shape[1], rng_c[-1], rng_c[0]])
        valid_c = np.isfinite(br_c)
        axes[0].plot(np.where(valid_c)[0], br_c[valid_c], 'r-', lw=1)
        axes[0].set_title(f'CRIMAC — {zp.name}')
        axes[0].set_ylabel('Range (m)')

        # echopype echogram + bottom
        sv_e = ep_data['sv_db'][0]
        rng_e = ep_data['range_array']
        br_e = ep_data['bottom_range']
        axes[1].imshow(sv_e, aspect='auto', origin='upper', vmin=-80, vmax=-34,
                       extent=[0, sv_e.shape[1], rng_e[-1], rng_e[0]])
        valid_e = np.isfinite(br_e)
        axes[1].plot(np.where(valid_e)[0], br_e[valid_e], 'r-', lw=1)
        axes[1].set_title(f'echopype — {zp.name}')

        # Bottom depth difference
        n_common = min(len(br_c), len(br_e))
        diff = br_c[:n_common] - br_e[:n_common]
        both_valid = np.isfinite(diff)
        if both_valid.any():
            axes[2].hist(diff[both_valid], bins=50, edgecolor='black', alpha=0.7)
            axes[2].axvline(0, color='red', ls='--')
            axes[2].set_xlabel('CRIMAC - echopype bottom depth (m)')
            axes[2].set_title(f'Median diff: {np.nanmedian(diff):.2f} m, '
                              f'MAD: {np.nanmedian(np.abs(diff[both_valid])):.2f} m')
        else:
            axes[2].text(0.5, 0.5, 'No overlapping valid pings',
                         ha='center', va='center', transform=axes[2].transAxes)

        plt.tight_layout()
        plt.show()
else:
    print(f'echopype output directory not found: {ECHOPYPE_DIR}')
    print('Run notebook 14 first, then re-run this cell.')